In [3]:

import numpy as np
import random
import math

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

# Load Dataset
data = load_breast_cancer()
X = data.data
y = data.target
features = data.feature_names

# Parameters
N = 10
ITER = 30
PA = 0.25
ALPHA = 0.1
D = X.shape[1]

# Fitness Function
def fitness(solution):
    mask = solution > 0.5

    if not np.any(mask):
        return 1.0

    model = make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=1000)
    )

    accuracy = cross_val_score(
        model, X[:, mask], y, cv=5
    ).mean()

    return 1 - accuracy + 0.01 * np.sum(mask) / D

# Levy Flight
def levy_flight():
    beta = 1.5
    sigma = (
        math.gamma(1 + beta) * math.sin(math.pi * beta / 2) /
        (math.gamma((1 + beta) / 2) * beta *
         2 ** ((beta - 1) / 2))
    ) ** (1 / beta)

    u = np.random.normal(0, sigma, D)
    v = np.random.normal(0, 1, D)

    return u / (np.abs(v) ** (1 / beta) + 1e-10)

# Initialize Nests
nests = np.random.rand(N, D)
scores = np.array([fitness(n) for n in nests])

best = nests[np.argmin(scores)].copy()
best_score = min(scores)

# Cuckoo Search
for iteration in range(ITER):

    for i in range(N):
        new = np.clip(nests[i] + ALPHA * levy_flight(), 0, 1)
        j = random.randrange(N)

        score = fitness(new)

        if score < scores[j]:
            nests[j] = new
            scores[j] = score

    # Abandon Worst Nests
    count = max(1, int(PA * N))
    worst = np.argsort(scores)[-count:]

    for i in worst:
        nests[i] = np.random.rand(D)
        scores[i] = fitness(nests[i])

    # Update Best
    idx = np.argmin(scores)

    if scores[idx] < best_score:
        best = nests[idx].copy()
        best_score = scores[idx]

# Selected Features
selected = best > 0.5

# Final Evaluation
model = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000)
)

accuracy = cross_val_score(
    model, X[:, selected], y, cv=5
).mean()

# Output
print("Selected Features:")
print(features[selected])

print("\nTotal Features:", D)
print("Selected Features:", np.sum(selected))
print("Final Accuracy:", round(accuracy * 100, 2), "%")


Selected Features:
['mean radius' 'mean texture' 'mean concavity' 'mean symmetry'
 'radius error' 'area error' 'concavity error' 'worst perimeter'
 'worst smoothness' 'worst compactness' 'worst concavity' 'worst symmetry'
 'worst fractal dimension']

Total Features: 30
Selected Features: 13
Final Accuracy: 98.24 %
